# 2번째로 1번째 과정에서 선택한 영화를 어느 지점에서 볼까 선택하는 과정

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime
from selenium import webdriver
import time

# 완성본
주토피아2 서울 상영 정보를 셀레니움으로 동적 크롤링 하는 코드

In [ ]:


def megabox_crawling():
    """메가박스 주토피아2 서울 지역 상영 정보 크롤링"""
    result = []
    wd = webdriver.Chrome()
    
    try:
        # 페이지 로드 및 iframe 전환 # 메가박스 페이지 구조상 페이지안에 또 다른 페이지가 있기에 iframe으로 전환해야 함
        wd.get("https://www.megabox.co.kr/booking") # 메가박스 예매 페이지(메인은 전 코드)
        time.sleep(3)
        
        iframe = wd.find_element("css selector", "iframe#frameBokdMBooking") # css selector로 iframe 요소 찾기
        wd.switch_to.frame(iframe)
        time.sleep(2)
        
        # 1. 주토피아2 선택
        for movie in wd.find_elements("css selector", "button[movie-nm]"): # css selector로 영화 버튼 요소들 찾기
            if "주토피아" in (movie.get_attribute("movie-nm") or ""): # 영화 이름이 "주토피아"인 버튼 클릭
                movie.click()
                time.sleep(1)
                break
        
        # 2. 29일 선택
        for date_btn in wd.find_elements("css selector", "button[date-data]"): # css selector로 날짜 버튼 요소들 찾기
            if "2025.11.29" in (date_btn.get_attribute("date-data") or ""): # 날짜가 "2025.11.29"인 버튼 클릭
                date_btn.click()
                time.sleep(2)
                break
        
        # 3. 서울 선택
        time.sleep(1)
        for city_btn in wd.find_elements("css selector", "div.all-list button.btn[id]"): # css selector로 도시 버튼(id) 요소들 찾기
            if "서울" in city_btn.text: # 버튼 텍스트에 "서울"이 포함된 버튼 클릭
                city_btn.click()
                time.sleep(2)
                break
        
        # 4. 극장 목록 스크롤 (모든 극장 로드)
        try:
            scroll_area = wd.find_element("css selector", "div.region-list") # css selector로 극장 목록이 있는 스크롤 영역 요소 찾기
            for _ in range(5):
                wd.execute_script("arguments[0].scrollTop += 300;", scroll_area) # 자바스크립트로 스크롤 영역을 아래로 300px씩 스크롤
                time.sleep(0.3)
            wd.execute_script("arguments[0].scrollTop = 0;", scroll_area) # 스크롤 영역을 맨 위로 이동
            time.sleep(0.5)
        except:
            pass
        
        # 5. 활성화된 극장 이름 목록 수집 (서울만 - 최대 18개)
        theater_names = []
        for btn in wd.find_elements("css selector", "button#btn[brch-nm]"): # css selector로 극장 버튼 요소들 찾기
            name = btn.get_attribute("brch-nm") # 버튼의 brch-nm 속성에서 극장 이름 추출
            if name and "disabled" not in (btn.get_attribute("class") or ""):
                theater_names.append(name)
        
        # 서울 극장만 (최대 18개)
        theater_names = theater_names[:18] # 서울 지역 극장 수는 최대 18개로 제한 -> 19번째(코엑스?) 부터 오류가 발생했음
        print(f"극장 {len(theater_names)}개: {theater_names}")
        
        # 6. 각 극장 순회하며 크롤링
        for idx, target in enumerate(theater_names): # 인덱스 번호와 값을 같이 가져오는 enumerate로 극장 이름 순회
            print(f"[{idx+1}/{len(theater_names)}] {target} 처리 중...")
            
            # 극장 클릭
            try:
                for btn in wd.find_elements("css selector", "button#btn[brch-nm]"): # css selector로 극장 버튼 요소들 찾기 # 극장 클릭할 때마다 페이지가 갱신되어서 이전에 저장한 버튼 객차가 무효화 될 수 있어서 매번 버튼 요소를 다시 찾기
                    if btn.get_attribute("brch-nm") == target: # 버튼의 brch-nm 속성이 현재 타겟 극장 이름과 일치하는지 확인
                        wd.execute_script("arguments[0].scrollIntoView({block:'center'});", btn) # 자바스크립트로 버튼이 화면 중앙에 오도록 스크롤
                        time.sleep(0.3)
                        btn.click()
                        time.sleep(2)
                        break
            except:
                continue
            
            # 상영 정보 파싱
            soup = BeautifulSoup(wd.page_source, 'html.parser') # 현재 페이지의 HTML 소스를 BeautifulSoup으로 파싱
            for item in soup.find_all("button", {"play-start-time": True}): # play-start-time 속성이 있는 버튼 요소들 찾기 (상영 정보가 담긴 버튼)
                time_el = item.find("span", class_="time") # 상영 시간 정보가 담긴 span 요소 찾기(밑에 전부 비슷)
                theater_el = item.find("span", class_="theater")
                seat_el = item.find("span", class_="seat")
                title_el = item.find("span", class_="title")
                
                # 시작/종료 시간
                start = time_el.find("strong").text.strip() if time_el and time_el.find("strong") else "" # 상영 시간 정보에서 strong 요소의 텍스트가 시작 시간, 없으면 빈 문자열
                time_text = time_el.get_text(strip=True) if time_el else "" # 상영 시간 정보의 전체 텍스트 (공백 제거)
                end = "~" + time_text.split("~")[1].split()[0] if "~" in time_text else "" # 상영 시간 텍스트에서 "~"로 구분된 부분이 있으면 종료 시간 추출, 없으면 빈 문자열
                
                # 상영관, 좌석, 타입
                screen = theater_el.get_text(strip=True) if theater_el else "" # 상영관 정보의 전체 텍스트 (공백 제거)
                current = seat_el.find("strong", class_="now").text.strip() if seat_el and seat_el.find("strong", class_="now") else "" # 좌석 정보에서 현재 잔여 좌석 수가 담긴 strong.now 요소의 텍스트, 없으면 빈 문자열
                total = seat_el.find("em", class_="all").text.strip() if seat_el and seat_el.find("em", class_="all") else "" # 좌석 정보에서 전체 좌석 수가 담긴 em.all 요소의 텍스트, 없으면 빈 문자열
                stype = title_el.find("em").text.strip() if title_el and title_el.find("em") else "" # 상영 타입 정보가 담긴 em 요소의 텍스트, 없으면 빈 문자열
                
                result.append(["서울", target, start, end, stype, screen, current, total])
            
            # 극장 선택 해제
            try: # 극장 버튼을 다시 찾아서 클릭하여 선택 해제 -> 메가박스 사이트 특 -> 위 극장 선택과정과 일치함
                for btn in wd.find_elements("css selector", "button#btn[brch-nm]"): 
                    if btn.get_attribute("brch-nm") == target: #
                        wd.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                        time.sleep(0.3)
                        btn.click()
                        time.sleep(1)
                        break
            except:
                time.sleep(1)
        
    finally:
        wd.quit()
    
    return result


def main():
    print("메가박스 크롤링 시작")
    result = megabox_crawling()
    
    if result:
        df = pd.DataFrame(result, columns=['도시', '극장', '시작시간', '종료시간', '상영타입', '상영관', '잔여좌석', '전체좌석'])
        df.to_csv('Megabox_Zootopia2_Seoul_1129.csv', encoding='utf-8-sig', index=True)
        print(f"\n총 {len(result)}개 수집 완료")
        print(df)
    else:
        print("수집된 데이터 없음")


if __name__ == '__main__':
    main()

메가박스 크롤링 시작
극장 18개: ['강남', '강동', '구의이스트폴', '군자', '더부티크목동현대백화점', '동대문', '마곡', '목동', '상봉', '상암월드컵경기장', '센트럴', '송파파크하비오', '신촌', '이수', '창동', '코엑스', '홍대', '화곡']
[1/18] 강남 처리 중...
[2/18] 강동 처리 중...
[3/18] 구의이스트폴 처리 중...
[4/18] 군자 처리 중...
[5/18] 더부티크목동현대백화점 처리 중...
[6/18] 동대문 처리 중...
[7/18] 마곡 처리 중...
[8/18] 목동 처리 중...
[9/18] 상봉 처리 중...
[10/18] 상암월드컵경기장 처리 중...
[11/18] 센트럴 처리 중...
[12/18] 송파파크하비오 처리 중...
[13/18] 신촌 처리 중...
[14/18] 이수 처리 중...
[15/18] 창동 처리 중...
[16/18] 코엑스 처리 중...
[17/18] 홍대 처리 중...
[18/18] 화곡 처리 중...

총 451개 수집 완료
     도시  극장   시작시간    종료시간    상영타입              상영관 잔여좌석 전체좌석
0    서울  강남  09:00  ~10:58  2D(자막)     강남르 리클라이너 1관  105  116
1    서울  강남  09:35  ~11:33  2D(자막)     강남르 리클라이너 3관   92  116
2    서울  강남  10:45  ~12:43  2D(더빙)     강남르 리클라이너 4관   16   53
3    서울  강남  11:20  ~13:18  2D(자막)     강남르 리클라이너 1관   68  116
4    서울  강남  11:50  ~13:48  2D(자막)     강남르 리클라이너 3관   78  116
..   ..  ..    ...     ...     ...              ...  ...  ...
446  서울  화곡  19:40  ~21:38  2D(자막)  